# STEP11. MRL + DMD 눈 데이터셋 통합

## 분석 질문

- MRL 과 DMD 를 어떤 표본 비율·어떤 분할로 합치면 Closed 표본 비율이 유지되는가.
- DMD의 Open 프레임 비중이 높은 특성을 고려하면서, 학습용 합본의 Closed 비율을 MRL과 유사하게 유지할 수 있는가.
- DMD subject가 학습·검증·평가에 걸쳐 중복되지 않도록 분할할 수 있는가.
- 이러한 데이터 구성과 분할이 실제 모델 성능에 어떤 영향을 주는지는 STEP12에서 평가한다.

## 두 데이터셋의 역할

| | 규모 | 성격 | 역할 |
|---|---|---|---|
| MRL Eye | 84,898장 / 37명 | IR 그레이스케일 눈 crop | **주 학습 데이터** |
| DMD | 16영상 / 13명 | RGB 주행 자세 영상 | 도메인 보강 + hold-out 평가 |

MRL 은 이미 `mrl_split.py` 로 subject 분할 manifest 가 만들어져 있어 이 노트북에서는 읽어서 검증만 한다. 코드 분량이 DMD 쪽에 쏠린 것은 DMD 만 영상에서 crop 을 새로 뽑아야 하기 때문이다.

## 입력

| 구분 | 경로 |
|---|---|
| MRL subject 분할 manifest | `config.OUTPUTS_DIR / "mrl_split" / "mrl_manifest_subject.csv"` |
| MRL 눈 이미지 | `build_dmd_eye_dataset.data_subdir("MRL Eye") / "data"` |
| DMD 프레임 GT (STEP10) | `config.OUTPUTS_DIR / "dmd_gt"` |
| DMD mosaic 영상 | `build_dmd_eye_dataset.dmd_dir()` |
| YuNet 모델 | `config.YUNET_MODEL` |

## 출력

| 파일 | 위치 |
|---|---|
| DMD 눈 crop png | `config.OUTPUTS_DIR / "dmd_eye" / <split> / <label>` |
| `dmd_eye_manifest.csv` | `config.OUTPUTS_DIR / "dmd_eye"` |
| `eye_manifest.csv` (통합) | `config.OUTPUTS_DIR / "eye_dataset"` |
| `class_balance.csv` · `leakage_report.csv` | `config.OUTPUTS_DIR / "eye_dataset"` |

## 전체 수행 흐름

**PART A — 진단**

1. 설정·경로
1b. MRL manifest 정합성 검증
2. MRL·DMD 클래스 구성
3. 단순 합본의 클래스 비율

**PART B — DMD 눈 crop 생성**

4. 분할·stride 확정값
5. 예상 표본 수 (영상 디코딩 전)
6. 눈 crop 생성

**PART C — 통합 manifest**

7. MRL + DMD 결합
8. 누수 검증
9. 경로 유효성 표본 검사

## PART A — 진단

### 목적

- MRL manifest 가 디스크와 어긋나지 않는지 먼저 확인한다.
- 두 데이터셋의 클래스 구성을 숫자로 보고, 조정 없이 합쳐도 되는지 판정한다.

In [1]:
# [셀 1] 설정 · 경로 (import·경로·시드는 여기서 한 번만)

# --- 저장소 루트 부트스트랩 (모든 노트북 공통, 수정 금지) ---
import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():          # VS Code Notebook
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):                        # IPython 커널 시작 폴더
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())           # 최후 수단

# config.py 와 requirements.txt 를 '둘 다' 가진 폴더만 저장소 루트로 인정한다.
_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

if _root is not None and Path(config.__file__).resolve().parent != _root:
    raise ImportError(f"의도하지 않은 config.py 가 import 되었습니다: {config.__file__}")
# --- 부트스트랩 끝 ---

# 프로젝트 모듈은 전부 src/ 에 평탄하게 둔다. 아직 패키지가 아니므로 sys.path 로 붙인다.
_src = str(config.PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import pandas as pd
import build_dmd_eye_dataset as B

SEED = 42

# 데이터셋 위치는 B.data_subdir 로 해석한다 (data/raw/<name> · data/<name> 둘 다 허용).
MRL_ROOT = B.data_subdir("MRL Eye") / "data"
MRL_MANIFEST = config.OUTPUTS_DIR / "mrl_split" / "mrl_manifest_subject.csv"
DMD_EYE_DIR = config.OUTPUTS_DIR / "dmd_eye"
EYE_DS_DIR = config.OUTPUTS_DIR / "eye_dataset"
EYE_DS_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_MANIFEST = EYE_DS_DIR / "eye_manifest.csv"

for _label, _p in [("MRL 이미지", MRL_ROOT), ("MRL manifest", MRL_MANIFEST),
                   ("DMD GT (STEP10)", config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv"),
                   ("YuNet", B.yunet_path())]:
    print(f"  [{'o' if Path(_p).exists() else 'X'}] {_label:16s} {config._rel(_p)}")

  [o] MRL 이미지          data\raw\MRL Eye\data
  [o] MRL manifest     outputs\mrl_split\mrl_manifest_subject.csv
  [o] DMD GT (STEP10)  outputs\dmd_gt\_summary.csv
  [o] YuNet            model\detectors\face_detection_yunet_2023mar.onnx


### 목적

- MRL manifest 가 디스크의 실제 이미지와 1:1로 맞는지, subject 누수가 없는지 확인한다.
- MRL 은 이 노트북에서 새로 만들지 않고 승계하므로, 승계 전 검증이 유일한 확인 지점이다.

In [2]:
# [셀 1b] MRL manifest 정합성 검증
# MRL 은 이 노트북에서 crop 을 새로 만들지 않고 기존 manifest 를 그대로 승계한다.
# 승계 전에 manifest 가 디스크와 어긋나지 않는지 확인한다. 어긋나면 이후 전부 틀어진다.
import pandas as pd

if not MRL_MANIFEST.exists():
    raise FileNotFoundError(
        f"{config._rel(MRL_MANIFEST)} 가 없습니다. 먼저 아래를 실행하세요:\n"
        f'  python src/mrl_split.py --mrl_root "{MRL_ROOT}" '
        f'--out_dir "{MRL_MANIFEST.parent}"')

_mrl = pd.read_csv(MRL_MANIFEST)
_disk = {p.relative_to(MRL_ROOT).as_posix() for p in MRL_ROOT.glob("**/*.png")}

mrl_checks = [
    ("manifest 행 수", len(_mrl)),
    ("디스크 png 수", len(_disk)),
    ("manifest 에만 있는 경로", len(set(_mrl.path) - _disk)),
    ("디스크에만 있는 경로", len(_disk - set(_mrl.path))),
    ("path 중복", int(_mrl.path.duplicated().sum())),
    ("label 과 class_idx 불일치", int(((_mrl.class_idx == 0) != (_mrl.label == "Closed")).sum())),
    ("eyestate 와 label 불일치", int(((_mrl.eyestate == 0) != (_mrl.label == "Closed")).sum())),
    ("2개 이상 split 에 걸친 subject", int((_mrl.groupby("subject")["split"].nunique() > 1).sum())),
]
for k, v in mrl_checks:
    print(f"  {k:32s} {v:,}")

_bad = [k for k, v in mrl_checks[2:] if v != 0]
if _bad:
    raise AssertionError(f"MRL manifest 정합성 실패: {_bad}")
print("\nMRL manifest 정합성 통과.")

  manifest 행 수                     84,898
  디스크 png 수                        84,898
  manifest 에만 있는 경로                0
  디스크에만 있는 경로                      0
  path 중복                          0
  label 과 class_idx 불일치            0
  eyestate 와 label 불일치             0
  2개 이상 split 에 걸친 subject         0

MRL manifest 정합성 통과.


In [4]:
# [셀 2] MRL·DMD 클래스 구성
mrl = pd.read_csv(MRL_MANIFEST)

mrl_tab = (mrl.groupby("split")
             .agg(subjects=("subject", "nunique"), images=("path", "size"),
                  closed=("class_idx", lambda s: int((s == 0).sum())))
             .reindex(["train", "val", "test"]))
mrl_tab["closed_pct"] = (mrl_tab.closed / mrl_tab.images * 100).round(1)
mrl_tab["glasses_pct"] = (mrl.groupby("split")["glasses"].mean() * 100).round(1)

dmd_gt = pd.read_csv(config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv")
dmd_plan = pd.DataFrame(B.plan())
dmd_gt_closed = int(dmd_plan.gt_closed.sum())
dmd_gt_open = int(dmd_plan.gt_open.sum())

print("=== MRL (subject 분할) ===")
print(mrl_tab)
print(f"\n=== DMD (프레임 단위 GT, 눈 2개 = crop 2장) ===")
print(f"close 프레임 {dmd_gt_closed:,} / open 프레임 {dmd_gt_open:,} "
      f"-> crop 상한 Closed {dmd_gt_closed*2:,} / Open {dmd_gt_open*2:,}")
print(f"DMD Closed 비율 : {dmd_gt_closed/(dmd_gt_closed+dmd_gt_open)*100:.1f}%"
      f"   (MRL 은 {mrl_tab.closed.sum()/mrl_tab.images.sum()*100:.1f}%)")

=== MRL (subject 분할) ===
       subjects  images  closed  closed_pct  glasses_pct
split                                                   
train        22   60264   29668        49.2         28.7
val           6   12479    6129        49.1         27.2
test          9   12155    6149        50.6         27.5

=== DMD (프레임 단위 GT, 눈 2개 = crop 2장) ===
close 프레임 8,827 / open 프레임 56,475 -> crop 상한 Closed 17,654 / Open 112,950
DMD Closed 비율 : 13.5%   (MRL 은 49.4%)


In [5]:
# [셀 3] 단순 합본(stride 없이 전부)의 클래스 비율
mrl_c, mrl_o = int((mrl.class_idx == 0).sum()), int((mrl.class_idx == 1).sum())
dmd_c, dmd_o = dmd_gt_closed * 2, dmd_gt_open * 2

naive = pd.DataFrame([
    dict(source="MRL", Closed=mrl_c, Open=mrl_o),
    dict(source="DMD (전부)", Closed=dmd_c, Open=dmd_o),
    dict(source="단순 합본", Closed=mrl_c + dmd_c, Open=mrl_o + dmd_o),
])
naive["total"] = naive.Closed + naive.Open
naive["closed_pct"] = (naive.Closed / naive.total * 100).round(1)

print(naive.to_string(index=False))

# 합본 기준 지분. 행마다 total 로 나누면 MRL 행에서 100% 를 넘는 무의미한 값이 나온다.
_all = mrl_c + mrl_o + dmd_c + dmd_o
print(f"\n단순 합본 {_all:,}장 중 DMD Open 이 {dmd_o:,}장 = {dmd_o/_all*100:.1f}%")
print(f"Closed 비율 : MRL 단독 {mrl_c/(mrl_c+mrl_o)*100:.1f}% -> 단순 합본 "
      f"{(mrl_c+dmd_c)/_all*100:.1f}%")

  source  Closed   Open  total  closed_pct
     MRL   41946  42952  84898        49.4
DMD (전부)   17654 112950 130604        13.5
   단순 합본   59600 155902 215502        27.7

단순 합본 215,502장 중 DMD Open 이 112,950장 = 52.4%
Closed 비율 : MRL 단독 49.4% -> 단순 합본 27.7%


### 관찰 결과

- MRL은 Closed **49.4%**로 Open과 거의 균형을 이룬다. 3개 split의 안경 비율도 27.2~28.7%로 큰 차이가 없다.
- DMD는 Closed **13.5%**로 Open 비중이 높으며, Open 프레임이 Closed보다 약 6.4배 많다.
- 두 데이터를 단순히 합치면 Closed 비율은 **27.7%**로 MRL 단독보다 낮아진다.
- 단순 합본 215,502장 중 DMD Open 프레임은 **112,950장(52.4%)**이다.

## PART B — DMD 눈 crop 생성

### 목적

- MRL 의 클래스 균형을 깨지 않는 DMD 표본 수를 정한다.
- 학습에 쓰지 않은 subject 만 test 로 남긴다.

### 결정 박스 2 — DMD 표본을 줄이는 방법

- 문제: DMD를 전부 사용하면 Open 프레임이 전체 표본의 절반 이상을 차지한다.
- 선택: **클래스별 비대칭 stride**를 적용한다. Closed 2 / Open 12.
- 근거: DMD에서 Closed 프레임이 상대적으로 적기 때문에 Closed는 더 촘촘하게 추출하고, Open은 상대적으로 큰 stride를 적용해 클래스 비율을 조정한다.
- Open 프레임은 연속 구간에서 유사한 프레임이 반복될 가능성이 높으므로, 모든 프레임을 사용하는 것보다 큰 stride를 적용하는 것이 저장량과 중복 표본을 줄이는 데 유리하다.
- `class_weight`를 사용하는 방법도 가능하지만, 본 단계에서는 표본 구성 자체에서 클래스 비율을 조정하는 방식을 선택했다.
- stride는 프레임 인덱스가 아니라 **클래스별 등장 순서**를 기준으로 적용한다.
- 이 설정은 STEP12의 모델 성능 비교 전에 사전 결정한 표본 구성이다.

### 결정 박스 3 — DMD subject 분할

- 문제: 13명을 train / val / test 로 나눈다. 기존 코드의 test 는 {gC_13, gF_23, gZ_37, gB_6} 였다.
- 선택: **test = {gE_29, gC_13, gB_6, gZ_37} / val = {gA_5, gF_23} / train = 나머지 7명.**
- 근거: 기존 test 에는 안경 착용자가 없어 안경 조건의 일반화를 확인할 수 없었다. 안경 착용자 3명을 train 2 / test 1 로 나누고 test 성별을 2:2 로 맞췄다. 2세션 subject(gB_10 · gZ_33 → train, gF_23 → val)는 같은 split 에 둔다.
- 안경 착용자가 3명뿐이라 이 배분은 통계적 검정용이 아니라 최소한의 확인 장치다.

### 결정 박스 4 — test의 stride

- 문제: test에서도 클래스별 비대칭 stride를 적용하면 클래스 비율이 인위적으로 조정될 수 있다.
- 선택: **비대칭 stride를 적용하지 않고 두 클래스에 동일한 stride(4)를 적용한다.**
- 근거: test에서는 학습용 클래스 균형을 맞추기보다, 원래 DMD annotation에서 관찰되는 클래스 구성의 비율을 유지하는 방향을 선택한다.
- 동일 stride를 적용하면 표본 수는 줄어들지만 클래스 간 상대적 구성은 임의로 조정하지 않는다.
- 대가: test Closed는 429프레임(858 crop)으로 줄어들어, Closed-Recall의 불확실성이 커지고 subject별 비교에는 한계가 있다.
- 더 많은 평가 표본이 필요한 경우 `STRIDE["test"]`를 2로 낮추는 대안을 검토할 수 있으나, 본 분석에서는 사전 설정한 stride 4를 유지한다.

In [6]:
# [셀 4] 분할·stride 확정값 확인 (build_dmd_eye_dataset 의 상수를 그대로 읽는다)
split_tab = (pd.DataFrame([dict(subject=k, split=v) for k, v in B.DMD_SPLIT.items()])
             .merge(dmd_gt[["subject", "video", "glasses", "gender"]], on="subject"))

print("=== stride (split x 클래스) ===")
print(pd.DataFrame(B.STRIDE).T.rename_axis("split"))
print("\n=== subject 분할 ===")
print(split_tab.groupby("split")
      .agg(subjects=("subject", "nunique"), videos=("video", "size"),
           glasses_subjects=("glasses", lambda s: int(s.sum())))
      .reindex(["train", "val", "test"]))
print("\n=== 다중 세션 subject 가 한 split 에 있는지 ===")
multi = split_tab.groupby("subject").filter(lambda g: len(g) > 1)
print(multi.groupby("subject")["split"].nunique().to_dict(), "(모두 1이어야 한다)")
split_tab.sort_values(["split", "subject"])[["split", "subject", "gender", "glasses", "video"]]

=== stride (split x 클래스) ===
       Closed  Open
split              
train       2    12
val         2    12
test        4     4

=== subject 분할 ===
       subjects  videos  glasses_subjects
split                                    
train         7       9                 2
val           2       3                 0
test          4       4                 1

=== 다중 세션 subject 가 한 split 에 있는지 ===
{'gB_10': 1, 'gF_23': 1, 'gZ_33': 1} (모두 1이어야 한다)


,split,subject,gender,glasses,video
14,test,gB_6,Male,False,gB_6_s5_2019-03-13T13;37;11+01;00
13,test,gC_13,Female,False,gC_13_s5_2019-03-12T10;03;00+01;00
12,test,gE_29,Female,True,gE_29_s5_2019-03-15T13;51;09+01;00
15,test,gZ_37,Male,False,gZ_37_s5_2019-04-29T12;06;35+02;00
0,train,gA_1,Male,True,gA_1_s5_2019-03-14T14;26;17+01;00
3,train,gB_10,Male,False,gB_10_s5_2019-03-12T10;35;20+01;00
4,train,gB_10,Male,False,gB_10_s5_2019-03-13T14;17;28+01;00
1,train,gB_7,Male,False,gB_7_s5_2019-03-13T13;55;52+01;00
2,train,gB_9,Male,False,gB_9_s5_2019-03-07T16;31;48+01;00
5,train,gC_14,Male,False,gC_14_s5_2019-03-12T09;18;58+01;00


In [7]:
# [셀 5] 예상 표본 수 — 영상을 디코딩하기 전에 균형을 먼저 확인한다
plan_df = pd.DataFrame(B.plan())
agg = (plan_df.groupby("split")
       .agg(videos=("video", "size"), gt_closed=("gt_closed", "sum"),
            gt_open=("gt_open", "sum"), crops_closed=("crops_closed", "sum"),
            crops_open=("crops_open", "sum"))
       .reindex(["train", "val", "test"]))
agg["crops_total"] = agg.crops_closed + agg.crops_open
agg["closed_pct"] = (agg.crops_closed / agg.crops_total * 100).round(1)

print("=== DMD crop 예상 상한 (YuNet 검출 100% 성공 가정) ===")
print(agg)

# MRL train 과 합쳤을 때 균형이 유지되는지 확인
_mc = int(mrl.query("split=='train'").class_idx.eq(0).sum())
_mo = int(mrl.query("split=='train'").class_idx.eq(1).sum())
_dc, _do = int(agg.loc["train", "crops_closed"]), int(agg.loc["train", "crops_open"])
print(f"\n합본 train : Closed {_mc+_dc:,} / Open {_mo+_do:,}"
      f"  -> Closed {(_mc+_dc)/(_mc+_dc+_mo+_do)*100:.1f}%"
      f"  (MRL 단독 {_mc/(_mc+_mo)*100:.1f}%)")

=== DMD crop 예상 상한 (YuNet 검출 100% 성공 가정) ===
       videos  gt_closed  gt_open  crops_closed  crops_open  crops_total  \
split                                                                      
train       9       5629    31898          5632        5326        10958   
val         3       1488    11079          1490        1848         3338   
test        4       1710    13498           858        6752         7610   

       closed_pct  
split              
train        51.4  
val          44.6  
test         11.3  

합본 train : Closed 35,300 / Open 35,922  -> Closed 49.6%  (MRL 단독 49.2%)


> 영상 디코딩 전에 GT 만으로 계산한 split별 예상 crop 수. train·val 은 비대칭 stride 로 균형에 근접하고 test 는 균일 stride 로 실전 비율을 유지한다.

### 관찰 결과

- YuNet 얼굴 검출 실패는 16개 영상에서 총 **1프레임**이었다(gB_10 1세션).
- 실제 생성량은 train **10,956개(Closed 51.4%)**, val **3,338개(44.6%)**, test **7,610개(11.3%)**로, GT 기반 사전 예상값과 일치했다.
- test 4개 영상의 Closed crop 수는 gB_6 88, gC_13 84, gZ_37 208, gE_29 478로 영상별 차이가 있었다.

### 목적

- 실제 crop 을 생성한다. 16개 영상을 전부 디코딩하므로 이 셀만 시간이 오래 걸린다.
- 처음에는 `BUILD_ALL = False` 로 한 영상만 확인하고, 정상이면 `True` 로 전체를 만든다.

In [8]:
# [셀 6] DMD 눈 crop 생성  ※ 실행 시간이 긴 셀
BUILD_ALL = True       # 동작 확인 후 True 로 바꿔 16개 전부 생성

jsons = B.dmd_jsons()
videos = jsons if BUILD_ALL else jsons[:1]
print(f"대상 영상 {len(videos)} / {len(jsons)}개"
      f"{'' if BUILD_ALL else '  (점검 모드 — 통합 manifest 를 만들려면 BUILD_ALL=True 필요)'}\n")

det = B.YuNetDetector()                      # 경로는 config.YUNET_MODEL. 02 와 같은 설정
dmd_manifest_path = B.build(videos, det, DMD_EYE_DIR)

대상 영상 16 / 16개

gA_1_s5_2019-03-14T14;26;17+01;00  [train] Closed   660 / Open   600  (검출실패 0)
gA_5_s5_2019-03-13T09;06;49+01;00  [val  ] Closed   550 / Open   516  (검출실패 0)
gB_10_s5_2019-03-12T10;35;20+01;00  [train] Closed  1050 / Open   564  (검출실패 1)
gB_10_s5_2019-03-13T14;17;28+01;00  [train] Closed   614 / Open   606  (검출실패 0)
gB_6_s5_2019-03-13T13;37;11+01;00  [test ] Closed    88 / Open  1928  (검출실패 0)
gB_7_s5_2019-03-13T13;55;52+01;00  [train] Closed   616 / Open   516  (검출실패 0)
gB_9_s5_2019-03-07T16;31;48+01;00  [train] Closed   646 / Open   556  (검출실패 0)
gC_13_s5_2019-03-12T10;03;00+01;00  [test ] Closed    84 / Open  2042  (검출실패 0)
gC_14_s5_2019-03-12T09;18;58+01;00  [train] Closed   478 / Open   682  (검출실패 0)
gE_29_s5_2019-03-15T13;51;09+01;00  [test ] Closed   478 / Open  1344  (검출실패 0)
gF_23_s5_2019-03-11T10;19;19+01;00  [val  ] Closed   402 / Open   700  (검출실패 0)
gF_23_s5_2019-03-14T13;49;53+01;00  [val  ] Closed   538 / Open   632  (검출실패 0)
gZ_33_s5_2019-04-04T09;29;07+

### 관찰 결과

- YuNet 얼굴 검출 실패는 16개 영상 합계 1프레임이다(gB_10 1세션).
- 실제 생성량은 train 10,956 (Closed 51.4%) / val 3,338 (44.6%) / test 7,610 (11.3%) 으로 예상값과 거의 같다.
- test 4개 영상의 Closed crop 은 gB_6 88 · gC_13 84 · gZ_37 208 · gE_29 478 로 영상 간 차이가 크다.

## PART C — 통합 manifest

### 목적

- MRL 과 DMD 를 manifest 1개로 합쳐 split 정책을 하나로 만든다.
- 학습 소스를 바꾸는 실험을 `source` 컬럼 필터만으로 표현한다.

### 결정 박스 5 — 두 데이터셋의 결합 형태

- 문제: MRL 경로는 MRL 루트 기준, DMD 경로는 `outputs/dmd_eye` 기준 상대경로다. (a) 실험마다 임시 CSV 를 만들어 절대경로로 합칠지, (b) 저장소 루트 기준 상대경로 manifest 1개로 통일할지.
- 선택: **(b).** 로더에는 `mrl_root = str(config.PROJECT_ROOT)` 를 넘긴다.
- 근거: (a)는 실험마다 임시 파일이 생겨 재현이 어렵고 CSV 에 개인 PC 경로가 남는다. (b)는 `mrl_dataset.make_dataset` 을 고치지 않고 그대로 쓸 수 있다. MRL 의 기존 subject 분할은 승계하고 재분할하지 않는다.
- 사전 계획.

In [8]:
# [셀 7] MRL + DMD -> 통합 manifest
COLUMNS = ["path", "source", "subject", "video", "frame", "side",
           "label", "class_idx", "glasses", "split"]

_mrl_prefix = MRL_ROOT.resolve().relative_to(config.PROJECT_ROOT).as_posix()
mrl_u = pd.DataFrame({
    "path": _mrl_prefix + "/" + mrl["path"].astype(str),   # 저장소 루트 기준으로 통일
    "source": "mrl", "subject": mrl["subject"], "video": "-", "frame": -1, "side": "-",
    "label": mrl["label"], "class_idx": mrl["class_idx"],
    "glasses": mrl["glasses"].astype(int), "split": mrl["split"],
})

# 셀 6 을 다시 돌리지 않아도 이 셀부터 재실행할 수 있게, 변수가 없으면 디스크에서 찾는다.
# crop 생성은 16개 영상 디코딩이라 비싸므로 한 번 만들면 재사용한다.
_dmd_man = globals().get("dmd_manifest_path") or (DMD_EYE_DIR / "dmd_eye_manifest.csv")
if _dmd_man is None or not Path(_dmd_man).exists():
    raise FileNotFoundError(
        f"DMD manifest 가 없습니다: {config._rel(DMD_EYE_DIR / 'dmd_eye_manifest.csv')}\n"
        "  셀 6 을 BUILD_ALL=True 로 실행하세요.")
dmd_u = pd.read_csv(_dmd_man)[COLUMNS]

if set(dmd_u.split.unique()) != {"train", "val", "test"}:
    raise ValueError(f"DMD split 이 3종이 아닙니다: {sorted(dmd_u.split.unique())}. "
                     "BUILD_ALL=True 로 16개 전부 생성해야 val/test 가 채워집니다.")

uni = pd.concat([mrl_u[COLUMNS], dmd_u], ignore_index=True)
uni.to_csv(UNIFIED_MANIFEST, index=False)

bal = (uni.groupby(["split", "source"])
       .agg(n=("path", "size"), closed=("class_idx", lambda s: int((s == 0).sum())),
            subjects=("subject", "nunique"))
       .reset_index())
bal["closed_pct"] = (bal.closed / bal.n * 100).round(1)
bal.to_csv(EYE_DS_DIR / "class_balance.csv", index=False)

print("저장 :", config._rel(UNIFIED_MANIFEST), f"({len(uni):,} rows)")
print()
_piv = bal.pivot(index="split", columns="source", values=["n", "closed_pct"])
_piv[("n", "dmd")] = _piv[("n", "dmd")].astype(int)     # pivot 이 float 으로 바꿔 놓는다
_piv[("n", "mrl")] = _piv[("n", "mrl")].astype(int)
print(_piv.reindex(["train", "val", "test"]))

저장 : outputs\eye_dataset\eye_manifest.csv (106,802 rows)

            n        closed_pct      
source    dmd    mrl        dmd   mrl
split                                
train   10956  60264       51.4  49.2
val      3338  12479       44.6  49.1
test     7610  12155       11.3  50.6


> 통합 manifest 의 split × source 별 표본 수와 Closed 비율. `source` 컬럼이 남아 있어 MRL only / DMD only / 합본 실험을 필터로 구성할 수 있다.

### 목적

- 학습·평가 사이에 정보가 새지 않는지 기계적으로 확인하고 결과를 CSV 로 남긴다.

In [9]:
# [셀 8] 누수 검증 — 하나라도 0 이 아니면 즉시 중단
checks = []

for src in ("mrl", "dmd"):
    sub = uni[uni.source == src]
    checks.append(dict(
        check=f"{src}: 2개 이상 split 에 등장하는 subject 수",
        value=int((sub.groupby("subject")["split"].nunique() > 1).sum())))

dmd_only = uni[uni.source == "dmd"]
checks.append(dict(check="dmd: 2개 이상 split 에 등장하는 video 수",
                   value=int((dmd_only.groupby("video")["split"].nunique() > 1).sum())))
checks.append(dict(check="dmd: 같은 (video, frame) 의 L/R 이 다른 split 에 있는 경우",
                   value=int((dmd_only.groupby(["video", "frame"])["split"]
                              .nunique() > 1).sum())))
checks.append(dict(check="mrl/dmd subject ID 충돌 수",
                   value=len(set(uni.query("source=='mrl'").subject)
                             & set(uni.query("source=='dmd'").subject))))
checks.append(dict(check="path 중복 수", value=int(uni.path.duplicated().sum())))
checks.append(dict(check="label 과 class_idx 불일치 수",
                   value=int(((uni.class_idx == 0) != (uni.label == "Closed")).sum())))

leak = pd.DataFrame(checks)
leak.to_csv(EYE_DS_DIR / "leakage_report.csv", index=False)
print(leak.to_string(index=False))

if leak.value.sum() != 0:
    raise AssertionError("누수 검증 실패. 위 표에서 0 이 아닌 항목을 확인하세요.")
print("\n누수 검증 통과 (모든 항목 0).")

                                          check  value
              mrl: 2개 이상 split 에 등장하는 subject 수      0
              dmd: 2개 이상 split 에 등장하는 subject 수      0
                dmd: 2개 이상 split 에 등장하는 video 수      0
dmd: 같은 (video, frame) 의 L/R 이 다른 split 에 있는 경우      0
                        mrl/dmd subject ID 충돌 수      0
                                      path 중복 수      0
                        label 과 class_idx 불일치 수      0

누수 검증 통과 (모든 항목 0).


In [10]:
# [셀 9] 경로 유효성 표본 검사 — 전체 확인은 비용이 크므로 층화 표본으로 본다
sample = (uni.groupby(["split", "source"], group_keys=False)
          .apply(lambda g: g.sample(min(len(g), 150), random_state=SEED),
                 include_groups=False))
missing = [p for p in sample.path if not (config.PROJECT_ROOT / p).exists()]

print(f"표본 {len(sample)}개 중 존재하지 않는 경로 : {len(missing)}")
if missing:
    print("예시 :", missing[:3])
    raise FileNotFoundError("manifest 경로가 디스크와 어긋납니다.")

print("\n=== 통합 manifest 최종 ===")
print(uni.groupby("split").agg(n=("path", "size"), subjects=("subject", "nunique"),
                               closed_pct=("class_idx", lambda s: round((s == 0).mean()*100, 1)))
      .reindex(["train", "val", "test"]))

표본 900개 중 존재하지 않는 경로 : 0

=== 통합 manifest 최종 ===
           n  subjects  closed_pct
split                             
train  71220        29        49.6
val    15817         8        48.2
test   19765        13        35.5


### 관찰 결과

- 누수 검증 6개 항목이 모두 0이다.
- 층화 표본 900개에서 manifest 경로와 디스크 파일이 모두 일치한다.
- 통합 manifest 는 106,802행이고 split별 subject 수는 train 29 / val 8 / test 13 이다(MRL + DMD 합계).
- 통합 test 의 Closed 비율 35.5% 는 MRL test(50.6%)와 DMD test(11.3%)가 섞인 값이다. STEP12 의 주 평가는 `source == "dmd"` 로 걸러서 수행한다.

## 해석

- MRL과 DMD의 split 및 파일 경로 정보를 하나의 manifest로 통합해, 이후 실험에서 `source` 조건만으로 MRL only / DMD only / MRL+DMD 구성을 재현할 수 있도록 했다.
- 비대칭 stride를 적용한 DMD train의 Closed 비율은 **51.4%**이며, MRL과 결합한 train에서도 **약 49.6%**로 MRL 단독과 유사한 수준을 유지했다.
- 따라서 합본 학습 데이터에서는 DMD의 RGB·주행 자세 환경을 포함하면서도 클래스 비율이 지나치게 한쪽으로 치우치지 않도록 구성했다.
- 이 데이터 구성이 실제 Closed-Recall 향상으로 이어지는지는 **STEP12의 독립적인 성능 평가에서 확인한다.**
- test는 원래 DMD의 클래스 구성 비율을 유지하는 방향으로 구성했기 때문에, Accuracy보다 **Closed-Recall을 주요 평가 지표로 사용하는 것이 적절하다.**


## 한계

- **시간적 상관**: 동일 영상에서 인접 프레임을 여러 개 포함하므로 프레임 수가 독립적인 표본 수를 의미하지는 않는다. 다만 subject 단위 split을 적용해 동일 subject의 프레임이 train/test에 동시에 포함되는 누수는 방지했다.
- **좌우 눈 상관**: 동일 프레임에서 추출한 좌·우 눈 crop은 서로 독립적인 표본으로 보기 어렵다. 따라서 STEP12에서는 프레임 단위 지표와 함께 subject 단위 결과를 함께 확인한다.
- **test 표본 크기**: test Closed는 429프레임으로, 전체 test 표본은 충분히 크지 않다. 특히 subject별로 나누면 개별 subject의 성능 비교에는 불확실성이 크다.
- **안경 표본**: DMD 안경 착용자는 3명뿐이며 train 2 / test 1로 구성했다. 따라서 안경 여부에 따른 성능 차이를 통계적으로 일반화할 수 없다.
- **도메인 차이**: MRL은 IR 그레이스케일, DMD는 RGB 주행 영상 기반이다. grayscale로 입력 채널을 통일하더라도 조명·촬영 각도·센서 등의 차이는 남는다.
- **검출기 의존성**: DMD 눈 crop은 YuNet 얼굴 검출에 성공한 프레임에서 생성된다. 이번 데이터에서는 총 1프레임의 검출 실패가 확인되었다.
- **연출 상황**: DMD 영상은 모두 `Car Stopped` 조건이므로 실제 주행 환경에서의 눈 상태 검출 성능으로 일반화하기에는 한계가 있다(STEP10).

## STEP11 요약

### Takeaway

- MRL과 DMD를 단순 결합하면 Closed 비율이 **49.4% → 27.7%**로 낮아져 DMD의 높은 Open 비중이 합본 데이터에 큰 영향을 준다.
- 이를 조정하기 위해 DMD train에는 **Closed 2 / Open 12의 비대칭 stride**를 적용했고, 합본 train의 Closed 비율을 **약 49.6%**로 유지했다.
- DMD subject 분할을 재구성하여 test에 안경 착용 subject 1명을 포함했으며, 동일 subject가 train·val·test에 중복되지 않도록 구성했다.
- test에는 클래스별 비대칭 stride를 적용하지 않고 동일 stride를 사용하여 DMD의 원래 클래스 구성 비율을 임의로 조정하지 않았다.
- 최종 통합 manifest는 **106,802행**이며, 누수 검증에서 확인한 6개 항목은 모두 0이었다.
- 이 단계에서는 데이터 구성과 분할을 확정한 것이며, **MRL+DMD 합본이 실제 눈 감김 검출 성능을 향상시키는지는 STEP12에서 평가한다.**